In [1]:
import pysam

In [4]:
FWD_PRIMER = "ACACAGGGAGGGGAACAT"
REV_PRIMER = "TGCCATGGTGGTTTGCT"
MAX_MISMATCH = 2

def match_with_mismatch(seq, primer, max_mm):
    if len(seq) < len(primer):
        return False
    mismatches = 0
    for a, b in zip(seq[:len(primer)], primer):
        if a != b:
            mismatches += 1
            if mismatches > max_mm:
                return False
    return True

bam = pysam.AlignmentFile("aligned.sorted.bam", "rb")
out = pysam.AlignmentFile("tagged.sorted.bam", "wb", template=bam)

for read in bam:

    if read.is_unmapped:
        out.write(read)
        continue

    seq = read.query_sequence

    if not read.is_reverse:
        if match_with_mismatch(seq, FWD_PRIMER, MAX_MISMATCH):
            read.set_tag("XP", "FWD", value_type="Z")

    else:
        if match_with_mismatch(seq, REV_PRIMER, MAX_MISMATCH):
            read.set_tag("XP", "REV", value_type="Z")

    out.write(read)

bam.close()
out.close()


  

[W::hts_idx_load3] The index file is older than the data file: aligned.sorted.bam.bai
